<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module12/Lab8.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 8 — QAOA for Portfolio Optimization (Instructor)
**Quantum Optimization and Simulation — QAOA Laboratory Series**

Instructor version with answer key.

**Format:** 10–15 minute instructor walkthrough + about 45–60 minutes of independent work.

**Notebook style:** Most code is supplied. Cells marked **YOUR TURN** contain a small value, line, or function for you to complete.

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Apply QAOA to a problem that is not Max-Cut.
- Interpret a qubit as a binary asset-selection decision.
- Build a small mean-variance portfolio optimization model.
- Solve it with Qiskit's high-level QAOA.

We use four hypothetical assets. A bit \(x_i=1\) means **select asset i**, and \(x_i=0\) means **do not select it**.

We require exactly two assets.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

## Part A — Define a small portfolio

In [ ]:
# Hypothetical expected returns (not real investment advice/data)
mu = np.array([0.12, 0.10, 0.07, 0.06])

sigma = np.array([
    [0.040, 0.018, 0.010, 0.008],
    [0.018, 0.035, 0.012, 0.009],
    [0.010, 0.012, 0.025, 0.006],
    [0.008, 0.009, 0.006, 0.020],
])

risk_factor = 0.5
budget = 2

print("Expected returns:", mu)
print("Covariance matrix:\n", sigma)

We minimize

\[
q\,x^T\Sigma x - \mu^T x
\]

subject to \(\sum_i x_i=2\).

Lower objective = better risk/return tradeoff under this chosen model.

## Part B — Build the QuadraticProgram

In [ ]:
from qiskit_optimization import QuadraticProgram

def build_portfolio_qp(mu, sigma, risk_factor, budget):
    qp = QuadraticProgram("4-asset portfolio")
    for i in range(4):
        qp.binary_var(name=f"x{i}")

    linear = {f"x{i}": -mu[i] + risk_factor * sigma[i, i] for i in range(4)}
    quadratic = {}

    for i in range(4):
        for j in range(i + 1, 4):
            quadratic[(f"x{i}", f"x{j}")] = 2 * risk_factor * sigma[i, j]

    qp.minimize(linear=linear, quadratic=quadratic)
    qp.linear_constraint(
        linear={f"x{i}": 1 for i in range(4)},
        sense="==",
        rhs=budget,
        name="choose_exactly_two"
    )
    return qp

qp = build_portfolio_qp(mu, sigma, risk_factor, budget)
print(qp.prettyprint())

## Part C — Classical brute-force reference

In [ ]:
from itertools import product

def portfolio_objective(x, risk_factor):
    x = np.asarray(x, dtype=float)
    return risk_factor * x @ sigma @ x - mu @ x

feasible = []
for x in product([0,1], repeat=4):
    if sum(x) == budget:
        feasible.append((x, portfolio_objective(x, risk_factor)))

feasible = sorted(feasible, key=lambda item: item[1])
feasible

**Expected:** six feasible portfolios, because there are \(inom{4}{2}=6\) ways to choose two of four assets.

## Part D — QAOA

In [ ]:
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.optimizers import COBYLA
from qiskit_aer.primitives import SamplerV2

qaoa = QAOA(
    sampler=SamplerV2(default_shots=4096, seed=SEED),
    optimizer=COBYLA(maxiter=80),
    reps=1
)

solver = MinimumEigenOptimizer(qaoa, penalty=2.0)
result = solver.solve(qp)
print(result.prettyprint())

selected = [i for i, value in enumerate(result.x) if round(value) == 1]
print("Selected assets:", selected)

### YOUR TURN
Change `risk_factor` from `0.5` to `2.0`, rebuild the problem with `build_portfolio_qp(...)`, and solve again.

**Question:** Does a stronger risk penalty change the selected pair? Explain using the covariance matrix.

## Final reflection
Complete this comparison:

| Max-Cut | Portfolio |
|---|---|
| qubit represents a ______ | qubit represents an ______ |
| bitstring represents a graph ______ | bitstring represents an asset ______ |
| objective counts cut ______ | objective balances expected ______ and ______ |

## Instructor solutions

**Interpretation:** Increasing the risk factor places greater weight on the covariance/risk term, so the optimizer may prefer a lower-return pair if it has sufficiently lower combined variance/covariance.

**Final reflection**
- qubit represents a **node**; qubit represents an **asset-selection decision**
- bitstring represents a graph **partition**; bitstring represents an asset **portfolio/selection**
- objective counts cut **edges**; objective balances expected **return** and **risk**